<a href="https://colab.research.google.com/github/khushnoodrehman/repo-exercise/blob/main/Chest_XRay.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
import os

# 1. kaggle.json file upload karein
print("Apni kaggle.json file upload karein:")
files.upload()

# 2. Kaggle API ko configure karein taake Colab usay access kar sake
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 3. 2GB wala Sample dataset download karein
!kaggle datasets download -d nih-chest-xrays/sample

# 4. Download hone ke baad us zip file ko extract (unzip) karein
!unzip -q sample.zip -d nih_sample_data
print("Dataset successfully download aur extract ho gaya hai!")

Apni kaggle.json file upload karein:


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/nih-chest-xrays/sample
License(s): CC0-1.0
100% 4.20G/4.20G [03:22<00:00, 22.3MB/s]

Dataset successfully download aur extract ho gaya hai!


In [2]:
import pandas as pd

# CSV file ko read karte hain
csv_path = '/content/nih_sample_data/sample_labels.csv'
df = pd.read_csv(csv_path)

# Total images aur shuru ki 5 rows print karein
print("Total X-Rays in dataset:", len(df))
print("\nDataset ki pehli 5 rows:")
# Hum sirf Image ka naam aur uski beemari wala column dekh rahe hain
print(df[['Image Index', 'Finding Labels']].head())

Total X-Rays in dataset: 5606

Dataset ki pehli 5 rows:
        Image Index                                     Finding Labels
0  00000013_005.png  Emphysema|Infiltration|Pleural_Thickening|Pneu...
1  00000013_026.png                             Cardiomegaly|Emphysema
2  00000017_001.png                                         No Finding
3  00000030_001.png                                        Atelectasis
4  00000032_001.png                        Cardiomegaly|Edema|Effusion


In [3]:
from sklearn.preprocessing import MultiLabelBinarizer

# 1. Jo labels '|' se jude hue hain, unko tod kar ek list bana lein
df['Labels_List'] = df['Finding Labels'].apply(lambda x: x.split('|'))

# 2. MultiLabelBinarizer use karke 0 aur 1 ke columns banayein
mlb = MultiLabelBinarizer()
labels_matrix = mlb.fit_transform(df['Labels_List'])

# 3. In naye 0/1 wale columns ko apne dataframe ka hissa bana lein
labels_df = pd.DataFrame(labels_matrix, columns=mlb.classes_)
df_encoded = pd.concat([df[['Image Index']], labels_df], axis=1)

print("Naye format ka Dataframe tayyar hai!\n")
print(df_encoded.head())

Naye format ka Dataframe tayyar hai!

        Image Index  Atelectasis  Cardiomegaly  Consolidation  Edema  \
0  00000013_005.png            0             0              0      0   
1  00000013_026.png            0             1              0      0   
2  00000017_001.png            0             0              0      0   
3  00000030_001.png            1             0              0      0   
4  00000032_001.png            0             1              0      1   

   Effusion  Emphysema  Fibrosis  Hernia  Infiltration  Mass  No Finding  \
0         0          1         0       0             1     0           0   
1         0          1         0       0             0     0           0   
2         0          0         0       0             0     0           1   
3         0          0         0       0             0     0           0   
4         1          0         0       0             0     0           0   

   Nodule  Pleural_Thickening  Pneumonia  Pneumothorax  
0       0      

In [4]:
!pip install iterative-stratification

In [14]:
import os
import shutil
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# STEP 1: Main folders banana (train, val, test)
base_dir = '/content/radify_dataset'
splits = ['train', 'val', 'test']

for split in splits:
    os.makedirs(os.path.join(base_dir, split), exist_ok=True)

print("✅ Folders ban gaye hain!")

import os
import shutil
import glob

print("🔍 Original images ka sahi rasta (path) dhoondh rahe hain...\n")

# Yeh line pure folder mein andar tak ja kar .png files dhoondegi
all_images = glob.glob('/content/nih_sample_data/**/*.png', recursive=True)

if len(all_images) == 0:
    print("❌ Dataset mein koi image nahi mili! Extract sahi nahi hua.")
else:
    # Pehli image jahan mili, us folder ka address nikal lo
    real_source_dir = os.path.dirname(all_images[0])
    print(f"✅ Asal folder mil gaya: {real_source_dir}\n")

    base_dir = '/content/radify_dataset'

    # Naya aur behtar copy function
    def copy_images_fixed(image_list, folder_name):
        copied_count = 0
        for img_name in image_list:
            src_path = os.path.join(real_source_dir, img_name)
            dst_path = os.path.join(base_dir, folder_name, img_name)

            # Agar image sahi address par hai, toh usay apne folder mein copy karo
            if os.path.exists(src_path):
                shutil.copy(src_path, dst_path)
                copied_count += 1
        return copied_count

    print("⏳ Images ab actual path se copy ho rahi hain (Isme 1-2 minute lag sakte hain)...")

    # Dobara copy command chalana
    train_c = copy_images_fixed(X_train, 'train')
    val_c = copy_images_fixed(X_val, 'val')
    test_c = copy_images_fixed(X_test, 'test')

    print("\n🎉 Copy process mukammal ho gaya!")
    print(f"📁 TRAIN folder mein aayi: {train_c} images")
    print(f"📁 VAL folder mein aayi:   {val_c} images")
    print(f"📁 TEST folder mein aayi:  {test_c} images")

✅ Folders ban gaye hain!
🔍 Original images ka sahi rasta (path) dhoondh rahe hain...

✅ Asal folder mil gaya: /content/nih_sample_data/sample/images

⏳ Images ab actual path se copy ho rahi hain (Isme 1-2 minute lag sakte hain)...

🎉 Copy process mukammal ho gaya!
📁 TRAIN folder mein aayi: 3913 images
📁 VAL folder mein aayi:   841 images
📁 TEST folder mein aayi:  852 images


In [15]:
import os

base_dir = '/content/radify_dataset'

print("🔍 Folders ko scan kiya ja raha hai...\n")

# Har folder mein files ginn-ne ka loop
for folder in ['train', 'val', 'test']:
    folder_path = os.path.join(base_dir, folder)

    # Check karein ke folder majood hai ya nahi
    if os.path.exists(folder_path):
        # Folder ke andar majood tamam files ki list banayein
        files = os.listdir(folder_path)
        print(f"📁 {folder.upper()} folder mein total images hain: {len(files)}")
    else:
        print(f"❌ {folder.upper()} folder nahi mila!")

print("\n✅ Scanning mukammal ho gayi!")

🔍 Folders ko scan kiya ja raha hai...

📁 TRAIN folder mein total images hain: 3913
📁 VAL folder mein total images hain: 841
📁 TEST folder mein total images hain: 852

✅ Scanning mukammal ho gayi!


In [16]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Hamare DataFrames ko Train, Val aur Test mein alag karna
# df_encoded hamare paas pehle se majood hai, hum usme se apne splits nikal lenge
train_df = df_encoded[df_encoded['Image Index'].isin(X_train)]
val_df = df_encoded[df_encoded['Image Index'].isin(X_val)]
test_df = df_encoded[df_encoded['Image Index'].isin(X_test)]

# Hamari 14 diseases ke naam (columns)
disease_columns = mlb.classes_.tolist()

# 2. ImageDataGenerator Setup (With Data Augmentation)
# Train data mein hum thodi variations (Augmentation) dalenge taake model better seekhe
train_datagen = ImageDataGenerator(
    rescale=1./255,           # Image pixels ko 0-1 ke darmiyan lana (Model ke liye zaroori hai)
    rotation_range=10,        # Image ko halka sa rotate karna
    zoom_range=0.1,           # Halka sa zoom in/out karna
    horizontal_flip=True      # Left/Right flip karna
)

# Validation aur Test data mein sirf Rescale hota hai, koi augmentation nahi!
val_test_datagen = ImageDataGenerator(rescale=1./255)

# 3. Model ke hisaab se settings
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224) # Medical ML models (jaise DenseNet) is size par best kaam karte hain

print("🚀 Training Data load ho raha hai...")
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory='/content/radify_dataset/train',
    x_col="Image Index",
    y_col=disease_columns,
    class_mode="raw",       # "raw" ka matlab hai ke hamare labels 0 aur 1 ki form mein seedhe dataframe se uthao
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

print("\n✅ Validation Data load ho raha hai...")
val_generator = val_test_datagen.flow_from_dataframe(
    dataframe=val_df,
    directory='/content/radify_dataset/val',
    x_col="Image Index",
    y_col=disease_columns,
    class_mode="raw",
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

🚀 Training Data load ho raha hai...
Found 3913 validated image filenames.

✅ Validation Data load ho raha hai...
Found 841 validated image filenames.


In [17]:
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

print("🧠 DenseNet121 Base Model download ho raha hai...")

# 1. Base Model Load Karna (include_top=False ka matlab hai hum iska purana dimagh nikal rahe hain)
base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# 2. Base model ki layers ko freeze karna (taake unki purani training kharab na ho)
for layer in base_model.layers:
    layer.trainable = False

# 3. Apna Naya Dimagh (Custom Layers) Lagana
x = base_model.output
x = GlobalAveragePooling2D()(x)       # Data ko summarize karna
x = Dropout(0.3)(x)                   # Overfitting se bachne ke liye (Ratta na maare)

# 4. Final Output Layer
# Note: Humari 14 diseases hain, isliye len(disease_columns) lagaya hai
# Activation 'sigmoid' rakha hai kyunki Multi-Label mein ek sath 2 beemariyan bhi ho sakti hain
predictions = Dense(len(disease_columns), activation='sigmoid')(x)

# 5. Final Model Tayyar Karna
model = Model(inputs=base_model.input, outputs=predictions)

# 6. Model ko Compile (Configure) karna
model.compile(
    optimizer='adam',
    loss='binary_crossentropy', # Multi-label ke liye HAMESHA binary_crossentropy use hota hai
    metrics=['accuracy', tf.keras.metrics.AUC(multi_label=True)] # AUC medical models ke liye best metric hai
)

print("\n✅ Radify Chest X-Ray Model successfully tayyar aur compile ho gaya hai!")

🧠 DenseNet121 Base Model download ho raha hai...
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step

✅ Radify Chest X-Ray Model successfully tayyar aur compile ho gaya hai!


In [18]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# 1. Callbacks ko define karna
checkpoint = ModelCheckpoint(
    filepath='/content/radify_chest_model_best.h5', # Best model yahan save hoga
    monitor='val_loss',                            # Validation loss ko check karta rahega
    save_best_only=True,                           # Sirf behtareen model save karega
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,                                    # Agar 3 epochs tak loss kam nahi hua toh training rok dega
    restore_best_weights=True,                     # Best model ki weights khud wapas load kar lega
    verbose=1
)

# 2. Asal Training Shuru Karna
EPOCHS = 10  # Shuru mein hum 10 rounds ke liye train karenge

print("🔥 Radify Model ki training shuru ho rahi hai! Grab a cup of tea ☕...")

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=[checkpoint, early_stop]
)

print("\n🎉 Alhamdullilah! Model ki training mukammal ho gayi!")

🔥 Radify Model ki training shuru ho rahi hai! Grab a cup of tea ☕...
Epoch 1/10
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4648 - auc: 0.4955 - loss: 0.2872
Epoch 1: val_loss improved from None to 0.21153, saving model to /content/radify_chest_model_best.h5



Epoch 1: finished saving model to /content/radify_chest_model_best.h5
123/123 ━━━━━━━━━━━━━━━━━━━━ 200s 1s/step - accuracy: 0.4968 - auc: 0.5194 - loss: 0.2447 - val_accuracy: 0.5470 - val_auc: 0.5935 - val_loss: 0.2115
Epoch 2/10
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 898ms/step - accuracy: 0.5110 - auc: 0.5208 - loss: 0.2282
Epoch 2: val_loss improved from 0.21153 to 0.20518, saving model to /content/radify_chest_model_best.h5



Epoch 2: finished saving model to /content/radify_chest_model_best.h5
123/123 ━━━━━━━━━━━━━━━━━━━━ 127s 1s/step - accuracy: 0.5211 - auc: 0.5509 - loss: 0.2215 - val_accuracy: 0.5458 - val_auc: 0.6386 - val_loss: 0.2052
Epoch 3/10
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 868ms/step - accuracy: 0.5117 - auc: 0.5736 - loss: 0.2175
Epoch 3: val_loss improved from 0.20518 to 0.20201, saving model to /content/radify_chest_model_best.h5



Epoch 3: finished saving model to /content/radify_chest_model_best.h5
123/123 ━━━━━━━━━━━━━━━━━━━━ 123s 997ms/step - accuracy: 0.5224 - auc: 0.5879 - loss: 0.2146 - val_accuracy: 0.5505 - val_auc: 0.6628 - val_loss: 0.2020
Epoch 4/10
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 868ms/step - accuracy: 0.5238 - auc: 0.6080 - loss: 0.2090
Epoch 4: val_loss improved from 0.20201 to 0.20055, saving model to /content/radify_chest_model_best.h5



Epoch 4: finished saving model to /content/radify_chest_model_best.h5
123/123 ━━━━━━━━━━━━━━━━━━━━ 122s 995ms/step - accuracy: 0.5201 - auc: 0.6112 - loss: 0.2118 - val_accuracy: 0.5493 - val_auc: 0.6763 - val_loss: 0.2006
Epoch 5/10
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 839ms/step - accuracy: 0.5280 - auc: 0.6412 - loss: 0.2075
Epoch 5: val_loss improved from 0.20055 to 0.19809, saving model to /content/radify_chest_model_best.h5



Epoch 5: finished saving model to /content/radify_chest_model_best.h5
123/123 ━━━━━━━━━━━━━━━━━━━━ 118s 963ms/step - accuracy: 0.5305 - auc: 0.6393 - loss: 0.2075 - val_accuracy: 0.5505 - val_auc: 0.6845 - val_loss: 0.1981
Epoch 6/10
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 878ms/step - accuracy: 0.5174 - auc: 0.6379 - loss: 0.2068
Epoch 6: val_loss did not improve from 0.19809
123/123 ━━━━━━━━━━━━━━━━━━━━ 122s 996ms/step - accuracy: 0.5285 - auc: 0.6580 - loss: 0.2059 - val_accuracy: 0.5505 - val_auc: 0.6898 - val_loss: 0.1988
Epoch 7/10
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 845ms/step - accuracy: 0.5276 - auc: 0.6515 - loss: 0.2074
Epoch 7: val_loss improved from 0.19809 to 0.19804, saving model to /content/radify_chest_model_best.h5



Epoch 7: finished saving model to /content/radify_chest_model_best.h5
123/123 ━━━━━━━━━━━━━━━━━━━━ 119s 972ms/step - accuracy: 0.5334 - auc: 0.6603 - loss: 0.2044 - val_accuracy: 0.5470 - val_auc: 0.6891 - val_loss: 0.1980
Epoch 8/10
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 845ms/step - accuracy: 0.5411 - auc: 0.6674 - loss: 0.2005
Epoch 8: val_loss did not improve from 0.19804
123/123 ━━━━━━━━━━━━━━━━━━━━ 125s 1s/step - accuracy: 0.5346 - auc: 0.6770 - loss: 0.2034 - val_accuracy: 0.5458 - val_auc: 0.6932 - val_loss: 0.2002
Epoch 9/10
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 843ms/step - accuracy: 0.5273 - auc: 0.6800 - loss: 0.2010
Epoch 9: val_loss did not improve from 0.19804
123/123 ━━━━━━━━━━━━━━━━━━━━ 119s 965ms/step - accuracy: 0.5295 - auc: 0.6829 - loss: 0.2026 - val_accuracy: 0.5351 - val_auc: 0.6894 - val_loss: 0.1993
Epoch 10/10
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 839ms/step - accuracy: 0.5253 - auc: 0.6847 - loss: 0.2007
Epoch 10: val_loss improved from 0.19804 to 0.19699, saving model to /co


Epoch 10: finished saving model to /content/radify_chest_model_best.h5
123/123 ━━━━━━━━━━━━━━━━━━━━ 118s 963ms/step - accuracy: 0.5280 - auc: 0.6942 - loss: 0.2012 - val_accuracy: 0.5470 - val_auc: 0.6932 - val_loss: 0.1970
Restoring model weights from the end of the best epoch: 10.

🎉 Alhamdullilah! Model ki training mukammal ho gayi!


In [19]:
print("🧪 Test Data load ho raha hai (Final Exam)...")

# Test data ke liye generator setup (Shuffle=False rakhna zaroori hai)
test_generator = val_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory='/content/radify_dataset/test',
    x_col="Image Index",
    y_col=disease_columns,
    class_mode="raw",
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("\n📊 Model ki final testing shuru...")

# Model ko test data par evaluate karna
results = model.evaluate(test_generator)

print("\n✅ Final Test Results (Unseen Data):")
print(f"Test Loss:     {results[0]:.4f}")
print(f"Test Accuracy: {results[1]:.4f}")
print(f"Test AUC:      {results[2]:.4f}")

🧪 Test Data load ho raha hai (Final Exam)...
Found 852 validated image filenames.

📊 Model ki final testing shuru...
27/27 ━━━━━━━━━━━━━━━━━━━━ 28s 1s/step - accuracy: 0.5293 - auc: 0.6727 - loss: 0.1998

✅ Final Test Results (Unseen Data):
Test Loss:     0.1998
Test Accuracy: 0.5293
Test AUC:      0.6727
